# Basetable Ultimate

One clean daily row per date (Jul 5 – Nov 4 2024).  
Target: `polymarket_trump_prob`  
Sources: Polymarket · Time · Bluesky · Reddit · Newspapers · Google Trends · Polls · Financials

| Section | Features |
|---|---|
| 1. Target | Polymarket Trump win probability |
| 2. Time Dimension | Days to election, weekend dummy, weekday, campaign week |
| 3. Bluesky | Volume/engagement + RoBERTa sentiment + NRCLex emotions per buzz group |
| 4. Reddit | Volume/engagement + RoBERTa sentiment + NRCLex emotions per buzz group |
| 5. Newspapers | Volume per leaning + VADER sentiment + NRCLex emotions per leaning |
| 6. Google Trends | Daily search interest for 12 topic keywords |
| 7. Polls | Daily aggregated poll numbers + 7-day rolling avg |
| 8. Financials | S&P500, Oil, VIX, bonds, USD + derived returns |
| 9. Merge | Left join everything on date |
| 10. Lags | Polymarket lags 1–5, price change lags 1–3, key signal lags |
| 11. Save | `Data/3_Gold/basetable_ultimate.csv` |

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path('../Functions').resolve()))
from buzz_column import add_buzz_labels

DATA   = Path('../Data')
BRONZE = DATA / '1_Bronze'
SILVER = DATA / '2_Silver'
GOLD   = DATA / '3_Gold'

DATE_MIN = pd.Timestamp('2024-07-05')
DATE_MAX = pd.Timestamp('2024-11-04')
DATES    = pd.date_range(DATE_MIN, DATE_MAX, freq='D')

EMOTION_COLS = ['fear', 'anger', 'anticipation', 'trust', 'surprise', 'sadness', 'disgust', 'joy']

print(f'Date window : {DATE_MIN.date()} → {DATE_MAX.date()}  ({len(DATES)} days)')

---
## 1. Target — Polymarket

In [ ]:
poly = pd.read_csv(
    BRONZE / 'Polymarket' / 'polymarket_win_probabilities.csv',
    parse_dates=['date']
)
poly['polymarket_trump_prob'] = poly['Trump (%)'] / 100
poly = (
    poly[['date', 'polymarket_trump_prob']]
    .drop_duplicates('date')
    .sort_values('date')
    .loc[lambda d: (d['date'] >= DATE_MIN) & (d['date'] <= DATE_MAX)]
    .reset_index(drop=True)
)
print(f'Polymarket: {len(poly)} rows  |  {poly["date"].min().date()} → {poly["date"].max().date()}')
print(poly['polymarket_trump_prob'].describe().round(3))

---
## 2. Time Dimension

In [ ]:
ELECTION_DAY = pd.Timestamp('2024-11-05')

time_features = pd.DataFrame({'date': DATES})

time_features['days_to_election'] = (ELECTION_DAY - time_features['date']).dt.days
time_features['is_weekend']       = time_features['date'].dt.dayofweek.isin([5, 6]).astype(int)
time_features['weekday']          = time_features['date'].dt.dayofweek   # 0=Mon … 6=Sun
time_features['campaign_week']    = ((time_features['date'] - DATE_MIN).dt.days // 7) + 1

print(f'Time features: {len(time_features)} rows, {len(time_features.columns)} cols')
print(time_features.columns.tolist())
time_features.head(7)

---
## 3. Bluesky

In [ ]:
# --- Volume & engagement ---
bsky_raw = pd.read_csv(
    SILVER / 'Bluesky' / 'bluesky_clean.csv',
    usecols=['timestamp', 'candidate', 'likes', 'reposts', 'replies', 'is_reply']
)
bsky_raw['date'] = (
    pd.to_datetime(bsky_raw['timestamp'], utc=True)
    .dt.normalize().dt.tz_localize(None)
)
bsky_raw = bsky_raw[(bsky_raw['date'] >= DATE_MIN) & (bsky_raw['date'] <= DATE_MAX)]

bsky_vol = bsky_raw.groupby('date').agg(
    bsky_total_posts  = ('candidate', 'count'),
    bsky_trump_posts  = ('candidate', lambda x: (x == 'TrumpBuzz').sum()),
    bsky_harris_posts = ('candidate', lambda x: (x == 'HarrisBuzz').sum()),
    bsky_avg_likes    = ('likes',     'mean'),
    bsky_avg_reposts  = ('reposts',   'mean'),
    bsky_reply_ratio  = ('is_reply',  'mean'),
).reset_index()

bsky_vol['bsky_trump_share']  = bsky_vol['bsky_trump_posts']  / bsky_vol['bsky_total_posts']
bsky_vol['bsky_harris_share'] = bsky_vol['bsky_harris_posts'] / bsky_vol['bsky_total_posts']
bsky_vol = bsky_vol.drop(columns=['bsky_trump_posts', 'bsky_harris_posts'])

print(f'Bluesky volume: {len(bsky_vol)} rows, {len(bsky_vol.columns)} cols')
print(bsky_vol.columns.tolist())

In [ ]:
# --- RoBERTa sentiment + NRCLex emotions per buzz group ---
bsky_sent = pd.read_csv(SILVER / 'Bluesky' / 'sentiment_bluesky.csv')
bsky_sent['date'] = pd.to_datetime(bsky_sent['date']).dt.normalize()
bsky_sent = bsky_sent[(bsky_sent['date'] >= DATE_MIN) & (bsky_sent['date'] <= DATE_MAX)]

frames = []
for group, prefix in [('TrumpBuzz', 'bsky_trump'), ('HarrisBuzz', 'bsky_harris'), ('ElectionBuzz', 'bsky_election')]:
    sub = bsky_sent[bsky_sent['buzz_group'] == group].groupby('date').agg(
        sent_avg  = ('roberta_sentiment', 'mean'),
        sent_std  = ('roberta_sentiment', 'std'),
        pos_share = ('roberta_pos',       'mean'),
        neg_share = ('roberta_neg',       'mean'),
        **{e: (e, 'mean') for e in EMOTION_COLS}
    )
    sub.columns = [f'{prefix}_{c}' for c in sub.columns]
    frames.append(sub)

bsky_sentiment = pd.concat(frames, axis=1).reset_index()
bsky_sentiment['bsky_sentiment_gap'] = (
    bsky_sentiment['bsky_trump_sent_avg'] - bsky_sentiment['bsky_harris_sent_avg']
)

print(f'Bluesky sentiment: {len(bsky_sentiment)} rows, {len(bsky_sentiment.columns)} cols')
print(bsky_sentiment.columns.tolist())

---
## 4. Reddit

In [ ]:
# --- Volume & engagement (posts only for score/upvote_ratio/num_comments) ---
posts = pd.read_parquet(
    SILVER / 'Reddit' / 'reddit_posts_clean.parquet',
    columns=['created_utc', 'candidate', 'subreddit', 'score', 'upvote_ratio', 'num_comments']
)
posts['date'] = (
    pd.to_datetime(posts['created_utc'], utc=True)
    .dt.normalize().dt.tz_localize(None)
)
posts = posts[(posts['date'] >= DATE_MIN) & (posts['date'] <= DATE_MAX)]

reddit_vol = posts.groupby('date').agg(
    reddit_total_posts        = ('candidate',    'count'),
    reddit_trump_posts        = ('candidate',    lambda x: (x == 'TrumpBuzz').sum()),
    reddit_harris_posts       = ('candidate',    lambda x: (x == 'HarrisBuzz').sum()),
    reddit_avg_score          = ('score',        'mean'),
    reddit_avg_upvote_ratio   = ('upvote_ratio', 'mean'),
    reddit_avg_comments       = ('num_comments', 'mean'),
    reddit_conservative_posts = ('subreddit',    lambda x: x.isin(['conservative', 'trump']).sum()),
    reddit_liberal_posts      = ('subreddit',    lambda x: x.isin(['liberal', 'democrats']).sum()),
).reset_index()

reddit_vol['reddit_trump_share']        = reddit_vol['reddit_trump_posts']        / reddit_vol['reddit_total_posts']
reddit_vol['reddit_harris_share']       = reddit_vol['reddit_harris_posts']       / reddit_vol['reddit_total_posts']
reddit_vol['reddit_conservative_share'] = reddit_vol['reddit_conservative_posts'] / reddit_vol['reddit_total_posts']
reddit_vol['reddit_liberal_share']      = reddit_vol['reddit_liberal_posts']      / reddit_vol['reddit_total_posts']
reddit_vol = reddit_vol.drop(columns=['reddit_trump_posts', 'reddit_harris_posts',
                                       'reddit_conservative_posts', 'reddit_liberal_posts'])

print(f'Reddit volume: {len(reddit_vol)} rows, {len(reddit_vol.columns)} cols')
print(reddit_vol.columns.tolist())

In [ ]:
# --- RoBERTa sentiment + NRCLex emotions per buzz group ---
# Requires 1.5_sentiment_analysis.ipynb to have been run first
REDDIT_SENT_PATH = SILVER / 'Reddit' / 'sentiment_reddit.csv'

if REDDIT_SENT_PATH.exists():
    reddit_sent_raw = pd.read_csv(REDDIT_SENT_PATH)
    reddit_sent_raw['date'] = pd.to_datetime(reddit_sent_raw['date']).dt.normalize()
    reddit_sent_raw = reddit_sent_raw[
        (reddit_sent_raw['date'] >= DATE_MIN) & (reddit_sent_raw['date'] <= DATE_MAX)
    ]

    frames = []
    for group, prefix in [('TrumpBuzz', 'reddit_trump'), ('HarrisBuzz', 'reddit_harris'), ('ElectionBuzz', 'reddit_election')]:
        sub = reddit_sent_raw[reddit_sent_raw['candidate'] == group].groupby('date').agg(
            sent_avg  = ('roberta_sentiment', 'mean'),
            sent_std  = ('roberta_sentiment', 'std'),
            pos_share = ('roberta_pos',       'mean'),
            neg_share = ('roberta_neg',       'mean'),
            **{e: (e, 'mean') for e in EMOTION_COLS}
        )
        sub.columns = [f'{prefix}_{c}' for c in sub.columns]
        frames.append(sub)

    reddit_sentiment = pd.concat(frames, axis=1).reset_index()
    reddit_sentiment['reddit_sentiment_gap'] = (
        reddit_sentiment['reddit_trump_sent_avg'] - reddit_sentiment['reddit_harris_sent_avg']
    )
    print(f'Reddit sentiment: {len(reddit_sentiment)} rows, {len(reddit_sentiment.columns)} cols')
    print(reddit_sentiment.columns.tolist())
else:
    print('WARNING: sentiment_reddit.csv not found — run Descriptive/1_reddit/1.5_sentiment_analysis.ipynb first')
    reddit_sentiment = None

---
## 5. Newspapers

In [ ]:
# --- Volume: article counts by leaning + buzz classification (consistent with Bluesky/Reddit) ---
news_raw = pd.read_csv(
    SILVER / 'Newspapers' / 'mediacloud_articles_clean.csv',
    usecols=['date', 'leaning', 'title_clean'],
    parse_dates=['date']
)
news_raw = news_raw[(news_raw['date'] >= DATE_MIN) & (news_raw['date'] <= DATE_MAX)]

# Apply same buzz labeler as Bluesky & Reddit → candidate: TrumpBuzz / HarrisBuzz / ElectionBuzz
news_raw = add_buzz_labels(news_raw, text_col='title_clean')

news_vol = news_raw.groupby('date').agg(
    news_total         = ('title_clean', 'count'),
    news_trump_count   = ('candidate',   lambda x: (x == 'TrumpBuzz').sum()),
    news_harris_count  = ('candidate',   lambda x: (x == 'HarrisBuzz').sum()),
    news_election_count= ('candidate',   lambda x: (x == 'ElectionBuzz').sum()),
    news_rep_count     = ('leaning',     lambda x: (x == 'Republican').sum()),
    news_dem_count     = ('leaning',     lambda x: (x == 'Democratic').sum()),
    news_cen_count     = ('leaning',     lambda x: (x == 'Center/Unknown').sum()),
).reset_index()

news_vol['news_trump_share']         = news_vol['news_trump_count']   / news_vol['news_total']
news_vol['news_harris_share']        = news_vol['news_harris_count']  / news_vol['news_total']
news_vol['news_attention_asymmetry'] = (
    (news_vol['news_trump_count'] - news_vol['news_harris_count']) / news_vol['news_total']
)

print(f'Newspaper volume: {len(news_vol)} rows, {len(news_vol.columns)} cols')
print(news_vol.columns.tolist())

In [ ]:
# --- Sentiment: VADER + NRCLex per leaning — already daily in Silver ---
# Columns: vader_compound_mean/std/pos_neg_share x _dem/_rep/_cen
#        + nrc_<emotion> x _dem/_rep/_cen  (36 feature cols total)
news_sent = pd.read_csv(
    SILVER / 'Newspapers' / 'sentiment_features_newspapers.csv',
    parse_dates=['date']
)
news_sent = news_sent[(news_sent['date'] >= DATE_MIN) & (news_sent['date'] <= DATE_MAX)]

# Derived cross-leaning gap (rep minus dem VADER)
if 'vader_compound_mean_rep' in news_sent.columns and 'vader_compound_mean_dem' in news_sent.columns:
    news_sent['news_vader_leaning_gap'] = (
        news_sent['vader_compound_mean_rep'] - news_sent['vader_compound_mean_dem']
    )

print(f'Newspaper sentiment: {len(news_sent)} rows, {len(news_sent.columns)} cols')
print(news_sent.columns.tolist())

---
## 6. Google Trends

In [ ]:
trends = pd.read_csv(
    BRONZE / 'google_trends' / 'trends_daily_stitched.csv',
    parse_dates=['date']
)
trends = trends[(trends['date'] >= DATE_MIN) & (trends['date'] <= DATE_MAX)]

# Rename all keyword cols to gt_ prefix with underscored lowercase
rename = {c: 'gt_' + c.lower().replace(' ', '_') for c in trends.columns if c != 'date'}
trends = trends.rename(columns=rename)

print(f'Google Trends: {len(trends)} rows, {len(trends.columns)} cols')
print(trends.columns.tolist())

---
## 7. Polls

In [ ]:
polls_raw = pd.read_csv(BRONZE / 'Polls' / 'wikipedia_polls.csv')
polls_raw['Date'] = polls_raw['Date'].astype(str).str.replace(r'\[.*?\]', '', regex=True).str.strip()
polls_raw['date'] = pd.to_datetime(polls_raw['Date'], errors='coerce')
polls_raw = polls_raw.dropna(subset=['date', 'Trump', 'Harris'])
polls_raw = polls_raw[(polls_raw['date'] >= DATE_MIN) & (polls_raw['date'] <= DATE_MAX)]

polls_daily = polls_raw.groupby('date').agg(
    poll_trump_avg  = ('Trump',  'mean'),
    poll_harris_avg = ('Harris', 'mean'),
    poll_margin     = ('Margin', 'mean'),
    poll_n_today    = ('Trump',  'count'),
).reset_index()

# Align to master date index, forward-fill days without polls
polls_daily = (
    pd.DataFrame({'date': DATES})
    .merge(polls_daily, on='date', how='left')
)
for col in ['poll_trump_avg', 'poll_harris_avg', 'poll_margin']:
    polls_daily[col] = polls_daily[col].ffill()
polls_daily['poll_n_today'] = polls_daily['poll_n_today'].fillna(0).astype(int)

# Rolling averages
polls_daily['poll_trump_7d_avg']  = polls_daily['poll_trump_avg'].rolling(7, min_periods=1).mean()
polls_daily['poll_harris_7d_avg'] = polls_daily['poll_harris_avg'].rolling(7, min_periods=1).mean()
polls_daily['poll_margin_7d_avg'] = polls_daily['poll_margin'].rolling(7, min_periods=1).mean()

print(f'Polls: {len(polls_daily)} rows, {len(polls_daily.columns)} cols')
print(polls_daily.columns.tolist())

---
## 8. Financials

In [ ]:
market = (
    pd.read_csv(BRONZE / 'Financials' / 'market.csv', parse_dates=['Date'])
    .rename(columns={'Date': 'date', 'SP500': 'sp500', 'Oil': 'oil',
                     'VIX': 'vix', 'TenYearBond': 'bond_10y', 'USDIndex': 'usd_index'})
    .sort_values('date')
)

market['sp500_return_1d'] = market['sp500'].pct_change(1)
market['sp500_return_7d'] = market['sp500'].pct_change(7)
market['sp500_vol_7d']    = market['sp500_return_1d'].rolling(7).std()
market['oil_return_1d']   = market['oil'].pct_change(1)
market['vix_change_1d']   = market['vix'].diff(1)

market = market[(market['date'] >= DATE_MIN) & (market['date'] <= DATE_MAX)].reset_index(drop=True)

print(f'Financials: {len(market)} rows, {len(market.columns)} cols')
print(market.columns.tolist())
print(f'Note: market data only covers {market["date"].min().date()} → {market["date"].max().date()}')

---
## 9. Merge

In [ ]:
bt = poly.copy()

sources = [
    (time_features,  'Time dimension'),
    (bsky_vol,       'Bluesky volume'),
    (bsky_sentiment, 'Bluesky sentiment'),
    (reddit_vol,     'Reddit volume'),
    (news_vol,       'Newspaper volume'),
    (news_sent,      'Newspaper sentiment'),
    (trends,         'Google Trends'),
    (polls_daily,    'Polls'),
    (market,         'Financials'),
]
if reddit_sentiment is not None:
    sources.insert(4, (reddit_sentiment, 'Reddit sentiment'))

for df_src, name in sources:
    bt = bt.merge(df_src, on='date', how='left')
    print(f'  + {name:<25}  {len(df_src.columns)-1:>3} cols  →  total {len(bt.columns)}')

print(f'\nShape       : {bt.shape}')
print(f'Date range  : {bt["date"].min().date()} → {bt["date"].max().date()}')
print(f'Missing (%):\n{bt.isna().mean().sort_values(ascending=False).head(10).mul(100).round(1).to_string()}')

---
## 10. Lag features

In [ ]:
bt = bt.sort_values('date').reset_index(drop=True)

# Polymarket level lags (for AR models)
for k in range(1, 6):
    bt[f'prob_lag{k}'] = bt['polymarket_trump_prob'].shift(k)

# Daily price change + its lags
bt['price_change'] = bt['polymarket_trump_prob'].diff()
for k in range(1, 4):
    bt[f'price_change_lag{k}'] = bt['price_change'].shift(k)

# Key signal lags (t-1)
lag1_cols = [
    'bsky_sentiment_gap',
    'news_attention_asymmetry',
    'news_vader_leaning_gap',
    'poll_margin',
    'gt_trump',
    'gt_kamala',
    'vix',
    'sp500_return_1d',
]
if reddit_sentiment is not None:
    lag1_cols.append('reddit_sentiment_gap')

for col in lag1_cols:
    if col in bt.columns:
        bt[f'{col}_lag1'] = bt[col].shift(1)

n_lag_cols = len([c for c in bt.columns if 'lag' in c or c == 'price_change'])
print(f'Lag features added: {n_lag_cols} cols')
print(f'Final shape: {bt.shape}')

---
## 11. Save

In [ ]:
OUT = GOLD / 'basetable_ultimate.csv'
bt.to_csv(OUT, index=False)

print(f'Saved → {OUT}')
print(f'Shape : {bt.shape}  ({bt.shape[1]-1} features + date)\n')

groups = [
    ('Target',          [c for c in bt.columns if 'polymarket' in c]),
    ('Time',            [c for c in bt.columns if c in ['days_to_election','is_weekend','weekday','campaign_week']]),
    ('Bluesky',         [c for c in bt.columns if c.startswith('bsky_')]),
    ('Reddit',          [c for c in bt.columns if c.startswith('reddit_')]),
    ('Newspapers vol',  [c for c in bt.columns if c.startswith('news_')]),
    ('Newspapers sent', [c for c in bt.columns if c.startswith('vader_') or c.startswith('nrc_')]),
    ('Google Trends',   [c for c in bt.columns if c.startswith('gt_')]),
    ('Polls',           [c for c in bt.columns if c.startswith('poll_')]),
    ('Financials',      [c for c in bt.columns if c in ['sp500','oil','vix','bond_10y','usd_index']
                         or any(s in c for s in ['return','vol_','change_1d'])]),
    ('Lags',            [c for c in bt.columns if 'lag' in c or c == 'price_change']),
]

print(f'{"Group":<20} {"N cols":>6}')
print('-' * 28)
for name, cols in groups:
    print(f'{name:<20} {len(cols):>6}')
print('-' * 28)
print(f'{"TOTAL":<20} {sum(len(c) for _, c in groups):>6}')